In [1]:
#!pip install llama-index llama-index-llms-huggingface \
#    llama-index-llms-huggingface-api \
#    llama-index-embeddings-huggingface \
#    transformers accelerate bitsandbytes


INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# Import core libraries
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

import os
import zipfile


In [3]:
# Upload the BBC zip manually
from google.colab import files

print("Upload your BBC_Full_Text_Document_Classification.zip file:")
uploaded = files.upload()

# Detect uploaded zip file
zip_filename = list(uploaded.keys())[0]
print("Uploaded file:", zip_filename)

# Extract zip
import zipfile
import os
import shutil

extracted_dir = "/content/bbc_dataset"
os.makedirs(extracted_dir, exist_ok=True)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extracted_dir)

print("Dataset extracted to:", extracted_dir)

# Create papers/ folder
papers_dir = "/content/papers"
os.makedirs(papers_dir, exist_ok=True)

# Auto-pick 2 .txt files
selected_files = []

for root, dirs, files in os.walk(extracted_dir):
    for f in files:
        if f.endswith(".txt") and len(selected_files) < 2:
            src = os.path.join(root, f)
            dst = os.path.join(papers_dir, f)
            shutil.copy(src, dst)
            selected_files.append(dst)

print("\nCopied files to papers/:")
for f in selected_files:
    print(" -", f)


Upload your BBC_Full_Text_Document_Classification.zip file:


Saving BBC_Full_Text_Document_Classification.zip to BBC_Full_Text_Document_Classification.zip
Uploaded file: BBC_Full_Text_Document_Classification.zip
Dataset extracted to: /content/bbc_dataset

Copied files to papers/:
 - /content/papers/375.txt
 - /content/papers/269.txt


In [4]:
documents = SimpleDirectoryReader("papers").load_data()
print("Loaded documents:", len(documents))


Loaded documents: 2


In [5]:
llm = HuggingFaceLLM(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    tokenizer_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    context_window=2048,
    max_new_tokens=256,
    device_map="auto"
)

print("LLM loaded successfully.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

LLM loaded successfully.


In [6]:
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [7]:
Settings.llm = llm
Settings.embed_model = embed_model

print("Global settings applied.")


Global settings applied.


In [8]:
index = VectorStoreIndex.from_documents(documents)
print("Index successfully created.")


Index successfully created.


In [9]:
query_engine = index.as_query_engine()

response = query_engine.query("What are the main points in the article?")
print("Answer:", response)


Answer: 1. Internet giant Yahoo has launched software to allow people to search e-mail and other files on their PCs.
2. The desktop search technology has been licensed from a US-based firm X1 Technologies.
3. Searching e-mail effectively is becoming increasingly important, especially as the amount of spam increases.
4. Yahoo's software can also work separately on the desktop, searching for music, photos and other files.
5. Users can search under a variety of criteria, including file name, size, date and time.
6. It doesn't yet incorporate web searching, although Yahoo has promised that future versions will allow users to search both web-based and desktop data.
7. Search engines are often the first port of call for users when they go onto the web.
8. The new foray into desktop search has rung alarm bells for human rights groups, concerned about the implications to privacy.
9. Search engines are just one of many features people would like but I'm suspicious of its usefulness.
10. More us

In [ ]:
q1 = query_engine.query("What is the tone of the article?")
q2 = query_engine.query("Summarize key events described.")
q3 = query_engine.query("Explain any technical terms found.")
q4 = query_engine.query("Give a summary of the document in 5 bullet points.")

print("\nTone:", q1)
print("\nKey events:", q2)
print("\nTechnical terms:", q3)
print("\n5 bullet points:", q4)
